## Variance Inflation Factor (VIF)

We compute VIF to detect multicollinearity and remove highly correlated predictors.


In [6]:
import pandas as pd
from sklearn.impute import SimpleImputer
from statsmodels.stats.outliers_influence import variance_inflation_factor


In [7]:
df=pd.read_csv('cleaned_data.csv')

Handling missing values

In [8]:
imputer = SimpleImputer(strategy="median")
df_imputed = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)

Checking Multicollinearity

In [9]:


X = df_imputed.drop(columns=["falls_with_injury"])
vif = pd.DataFrame()
vif["variable"] = X.columns
vif["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print(vif)


                   variable         VIF
0            num_facilities    6.732119
1            overall_rating  278.906292
2             health_rating  201.726893
3           staffing_rating   40.168116
4            quality_rating   45.733979
5         nurse_hours_total   87.841048
6            rn_hours_total   14.458090
7        staff_turnover_pct   30.949907
8               total_fines    7.944479
9     total_payment_denials    3.280041
10  hosp_per_1000_residents   20.996786
11          pressure_ulcers   10.226588
12                  uti_pct    4.747574
13             catheter_pct    2.576445
14     mobility_decline_pct   33.597436
15         adl_increase_pct   33.199340
16           restrained_pct    1.215504
17         incontinence_pct   17.758743
18          weight_loss_pct   11.324483
19           depression_pct    2.375365


Removing high colinearity values

In [10]:
vars_to_drop = [
    'health_rating',
    'staffing_rating',
    'quality_rating',
    'nurse_hours_total',
    'adl_increase_pct'
]

df_reduced = df.drop(columns=vars_to_drop)

print("Remaining variables:", df_reduced.columns.tolist())
print("Shape after dropping:", df_reduced.shape)


Remaining variables: ['num_facilities', 'overall_rating', 'rn_hours_total', 'staff_turnover_pct', 'total_fines', 'total_payment_denials', 'hosp_per_1000_residents', 'falls_with_injury', 'pressure_ulcers', 'uti_pct', 'catheter_pct', 'mobility_decline_pct', 'restrained_pct', 'incontinence_pct', 'weight_loss_pct', 'depression_pct']
Shape after dropping: (618, 16)


Recalculating   VIF

In [15]:
imputer = SimpleImputer(strategy="median")
df_imputed = pd.DataFrame(imputer.fit_transform(df_reduced), columns=df_reduced.columns)

# Now run VIF safely
from statsmodels.stats.outliers_influence import variance_inflation_factor

X = df_imputed.drop(columns=['falls_with_injury'])

vif_df = pd.DataFrame()
vif_df['variable'] = X.columns
vif_df['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

print(vif_df.sort_values(by='VIF', ascending=False))
df_imputed.to_csv("df_final.csv", index=False)


                   variable        VIF
3        staff_turnover_pct  19.698296
12         incontinence_pct  16.314191
1            overall_rating  15.551236
6   hosp_per_1000_residents  15.267638
13          weight_loss_pct  10.922889
10     mobility_decline_pct  10.896168
7           pressure_ulcers   9.707209
2            rn_hours_total   7.916274
4               total_fines   7.846263
0            num_facilities   6.692503
8                   uti_pct   4.091252
5     total_payment_denials   3.242637
9              catheter_pct   2.517306
14           depression_pct   1.959321
11           restrained_pct   1.175549


### Variance Inflation Factor (VIF) Analysis

Variance Inflation Factor (VIF) was computed to assess multicollinearity among the 
predictor variables. The initial VIF results showed extremely high multicollinearity 
among several variables, particularly the facility rating metrics and certain staffing 
and functional decline measures. Notable high-VIF variables included:

- overall_rating (VIF ≈ 279)
- health_rating (VIF ≈ 202)
- quality_rating (VIF ≈ 46)
- staffing_rating (VIF ≈ 40)
- nurse_hours_total (VIF ≈ 88)
- mobility_decline_pct (VIF ≈ 34)
- adl_increase_pct (VIF ≈ 33)
- staff_turnover_pct (VIF ≈ 31)

These values indicate severe redundancy and instability if all variables were retained 
in the model. To address this, the following variables were removed:

- health_rating  
- staffing_rating  
- quality_rating  
- nurse_hours_total  
- adl_increase_pct  

The rationale for removal was based on domain logic and redundancy:
- The four rating variables were highly correlated; **overall_rating** was retained as 
  the most interpretable and comprehensive measure.
- nurse_hours_total and rn_hours_total were strongly collinear; **rn_hours_total** was 
  retained because it is more clinically meaningful.
- adl_increase_pct and mobility_decline_pct were highly correlated; **mobility_decline_pct** 
  was retained due to stronger clinical relevance.

After removing these variables, VIF was recalculated. All remaining predictors showed 
substantially reduced VIF values, with none exceeding the commonly accepted threshold 
of 10. This confirms that multicollinearity was successfully mitigated, resulting in a 
more stable and interpretable feature set for modeling.

The cleaned and reduced dataset was saved as `df_final.csv` for use in the modeling 
workflow.
